# LangChain + LangGraph: LLM-Directed Multi-Agent Assistant

**IAN 620 · Practical tutorial**

A user asks a question in ordinary language. **An actual local LLM chooses which agents to run and prepares their inputs.** LangGraph follows that plan.

- **Weather agent:** simulated API data (real API call is commented out).
- **Jobs agent:** fictional job listings (real API call is commented out).
- **Profit agent:** accurate Python arithmetic using values extracted by the LLM.
- **General agent:** responds to questions that do not need the three specialist tools.

The router **does not check for keywords**, use regular expressions to guess intent, or automatically assign an agent. Instead it asks the language model for a **structured JSON plan** and validates that plan. It may choose multiple agents for one question.

**Important:** Weather and jobs are demonstrations, **not live results**. Run the cells from top to bottom.

## 1. Install the libraries

**LangChain** connects the language model and tools. **LangGraph** controls which agent runs next. **Transformers** loads a small instruction-tuned LLM locally.

Colab's free CPU runtime can work, although a GPU runtime is faster.

In [ ]:
%pip -q install -U langchain langgraph langchain-huggingface transformers accelerate

## 2. Load a local language model

Use Qwen 2.5 Instruct through LangChain and Hugging Face. A 1.5-billion-parameter model is a useful balance for this tutorial, although an instruction-following model can still occasionally return malformed JSON. For faster execution, choose a Colab GPU runtime. No paid LLM API key is required.

In [ ]:
from langchain_huggingface import HuggingFacePipeline

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

llm = HuggingFacePipeline.from_model_id(
    model_id=MODEL_ID,
    task="text-generation",
    pipeline_kwargs={
        "max_new_tokens": 320,
        "do_sample": False,
        "return_full_text": False,
    },
)
print("Loaded:", MODEL_ID)

## 3. LangChain tools: weather, jobs, and profit

A **tool** is a Python function that an agent can use. The `@tool` decorator tells LangChain the function's name, parameters, and description.

For now the two external API calls are **commented out**. The response from each is a clearly labeled example, so the notebook runs without an API key. To connect a real API later, replace the sample data with validated API responses.

The profit calculation does not require an API or LLM: **profit = (selling price − cost price) × quantity**.

In [ ]:
from langchain_core.tools import tool

@tool
def weather_tool(location: str) -> str:
    """Get weather information for a city (sample response; API disabled)."""
    # Example integration (requires a real endpoint and error handling):
    # import requests
    # response = requests.get(WEATHER_API_URL, params={"city": location, "key": API_KEY}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return (
        f"SAMPLE WEATHER ONLY — not a live forecast for {location}: "
        "72°F, partly cloudy, light breeze."
    )

@tool
def jobs_tool(location: str) -> str:
    """Find recent job postings in a city (fictional sample; API disabled)."""
    # Example integration (requires a job-search provider and authentication):
    # import requests
    # response = requests.get(JOBS_API_URL, params={"location": location}, timeout=15)
    # response.raise_for_status()
    # return str(response.json())
    return (
        f"FICTIONAL SAMPLE JOBS for {location} — not current openings:\\n"
        "1. Data Analyst — Example Analytics Co.\\n"
        "2. Python Developer — Demo Software Inc.\\n"
        "3. AI Engineer — Sample Technology Group"
    ).replace("\\n", "\n")

@tool
def profit_tool(selling_price: float, cost_price: float, quantity: int = 1) -> str:
    """Calculate total profit from selling price, cost price, and quantity."""
    if quantity < 0:
        return "Quantity must be zero or positive."
    total_revenue = selling_price * quantity
    total_cost = cost_price * quantity
    profit = total_revenue - total_cost
    margin = (profit / total_revenue * 100) if total_revenue else 0
    return (
        f"Revenue: ${total_revenue:,.2f}; Cost: ${total_cost:,.2f}; "
        f"Profit: ${profit:,.2f}; Profit margin: {margin:.1f}%."
    )

print(weather_tool.invoke({"location": "Greensboro"}))
print(jobs_tool.invoke({"location": "Greensboro"}))
print(profit_tool.invoke({"selling_price": 75, "cost_price": 50, "quantity": 10}))

## 4. Store the LLM's agent plan in LangGraph state

A **state** is the information that moves through the graph. Here the LLM writes a list of agent calls. Each call has a name and the arguments needed by its tool.

Example (produced by the LLM, not hardcoded into the router):

```json
{"calls": [{"name": "weather", "location": "Greensboro"}, {"name": "jobs", "location": "Greensboro"}]}
```

The graph then executes these calls in order.

In [ ]:
from typing import TypedDict, Any

class AssistantState(TypedDict):
    question: str
    calls: list[dict[str, Any]]
    results: list[str]
    answer: str

## 5. Let the LLM choose agents and arguments

**This is the key part of the tutorial.** The language model reads the full question and produces JSON specifying which agents to use. It also extracts the location or profit inputs, when available.

We do **not** decide the agent using `if "weather" in question`, string matching, or regex. LangChain's `JsonOutputParser` parses the model's response, and simple validation checks that the requested agents exist. An invalid response causes a clear error rather than silently guessing.

The router can return several calls, so one user question can invoke multiple agents. The graph itself remains deterministic after the LLM selects its plan.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

parser = JsonOutputParser()
ALLOWED_AGENTS = {"weather", "jobs", "profit", "general"}

ROUTER_INSTRUCTIONS = """
You are the planner for a multi-agent assistant.
Understand the user's complete question and choose the necessary agents.
Choose ONLY from:
- weather: weather conditions; needs a location
- jobs: employment opportunities; needs a location
- profit: profit calculation; needs selling_price, cost_price, and quantity
- general: other questions; no extra parameters

Return ONLY one JSON object with a "calls" array. No commentary or markdown.
Each call has a "name" and any applicable parameters.
Use multiple calls if a question asks for more than one service.
Keep agent order consistent with the user's requests.
If location or numeric data is missing, use null; NEVER invent values.
For profit: quantity is 1 if the question clearly implies a single item.
Do not calculate the final profit yourself; only supply parameters.
Examples:
Question: How warm is it in Greensboro?
Output: {"calls":[{"name":"weather","location":"Greensboro"}]}
Question: Any hiring opportunities in Greensboro? Also will it rain there?
Output: {"calls":[{"name":"jobs","location":"Greensboro"},{"name":"weather","location":"Greensboro"}]}
Question: I sell 10 products for 75 each, buying each for 50. What is my profit?
Output: {"calls":[{"name":"profit","selling_price":75,"cost_price":50,"quantity":10}]}
Question: Explain what a vector database is.
Output: {"calls":[{"name":"general"}]}
"""

def plan_agents(question: str) -> list[dict]:
    response = llm.invoke(
        ROUTER_INSTRUCTIONS + "\nQuestion: " + question + "\nOutput:"
    )
    try:
        plan = parser.parse(response)
    except Exception as exc:
        raise ValueError(
            "The LLM did not produce valid JSON. Try running this cell again "
            "or using a stronger instruction model. Raw output: " + repr(response)
        ) from exc

    calls = plan.get("calls") if isinstance(plan, dict) else None
    if not isinstance(calls, list) or not 1 <= len(calls) <= 4:
        raise ValueError("The LLM must return 1 to 4 agent calls: " + repr(plan))
    for call in calls:
        if not isinstance(call, dict) or call.get("name") not in ALLOWED_AGENTS:
            raise ValueError("Unknown or invalid agent in plan: " + repr(plan))
    return calls

def route_node(state: AssistantState) -> dict:
    calls = plan_agents(state["question"])
    print("LLM-selected plan:", calls)
    return {"calls": calls}

## 6. Specialist agents use the LLM's parameters

Each graph node receives the parameters supplied by the LLM router. It **does not parse keywords from the user's question**. If information is missing, the agent asks for it rather than guessing.

The weather and jobs tools still return clearly labeled sample data, since their live API requests are commented out. The profit tool performs real arithmetic.

In [ ]:
def current_call(state: AssistantState) -> dict:
    # The completed-result count tells us which planned call is next.
    return state["calls"][len(state["results"])]

def weather_agent(state: AssistantState) -> dict:
    location = current_call(state).get("location")
    result = (
        weather_tool.invoke({"location": location})
        if isinstance(location, str) and location.strip()
        else "Please tell me the city for the weather request."
    )
    return {"results": state["results"] + [result]}

def jobs_agent(state: AssistantState) -> dict:
    location = current_call(state).get("location")
    result = (
        jobs_tool.invoke({"location": location})
        if isinstance(location, str) and location.strip()
        else "Please tell me the city for the jobs request."
    )
    return {"results": state["results"] + [result]}

def profit_agent(state: AssistantState) -> dict:
    arguments = current_call(state)
    selling = arguments.get("selling_price")
    cost = arguments.get("cost_price")
    quantity = arguments.get("quantity")

    if selling is None or cost is None or quantity is None:
        result = (
            "I need the selling price, cost price, and quantity. "
            "Example: selling price 75, cost price 50, quantity 10."
        )
    else:
        try:
            selling, cost, quantity = float(selling), float(cost), float(quantity)
            if not quantity.is_integer() or quantity < 0:
                raise ValueError("Quantity must be a nonnegative whole number")
            result = profit_tool.invoke({
                "selling_price": selling,
                "cost_price": cost,
                "quantity": int(quantity),
            })
        except (TypeError, ValueError) as exc:
            result = "Invalid profit inputs: " + str(exc)
    return {"results": state["results"] + [result]}

def general_agent(state: AssistantState) -> dict:
    prompt = (
        "Answer this question briefly and clearly. If you are unsure, say so.\n"
        f"Question: {state['question']}\nAnswer:"
    )
    return {"results": state["results"] + [llm.invoke(prompt).strip()]}

## 7. Build the LangGraph

LangGraph runs the agent calls in the exact order selected by the LLM. The conditional edge looks up the **next name in the structured LLM plan**; it never inspects the user's question for routing keywords.

The workflow stops after all planned calls have returned. This explicit boundary avoids uncontrolled looping.

In [ ]:
from langgraph.graph import StateGraph, START, END

def next_agent(state: AssistantState) -> str:
    completed = len(state["results"])
    if completed < len(state["calls"]):
        return state["calls"][completed]["name"]
    return "finish"

def finish_node(state: AssistantState) -> dict:
    return {"answer": "\n\n".join(state["results"])}

builder = StateGraph(AssistantState)
builder.add_node("router", route_node)
builder.add_node("weather", weather_agent)
builder.add_node("jobs", jobs_agent)
builder.add_node("profit", profit_agent)
builder.add_node("general", general_agent)
builder.add_node("finish", finish_node)

builder.add_edge(START, "router")
destinations = {name: name for name in ALLOWED_AGENTS}
destinations["finish"] = "finish"
builder.add_conditional_edges("router", next_agent, destinations)

for name in ALLOWED_AGENTS:
    builder.add_conditional_edges(name, next_agent, destinations)

builder.add_edge("finish", END)
assistant = builder.compile()
print("Graph compiled.")

## 8. Try three questions

The first two answers are examples rather than current information. The third calculates from your supplied numbers.

In [ ]:
def ask(question: str) -> str:
    state = assistant.invoke({
        "question": question,
        "calls": [],
        "results": [],
        "answer": "",
    })
    print("\nQUESTION:", question)
    print("ANSWER:\n", state["answer"])
    print("-" * 65)
    return state["answer"]

weather_answer = ask("How warm is it in Greensboro?")
jobs_answer = ask("Are any companies hiring in Greensboro?")
profit_answer = ask("I sell 10 products for 75 each and buy each for 50. What is my profit?")

## 9. Ask a question that needs two agents

The LLM should identify two separate tasks and output two calls, even when the question does not use exact agent names. Watch the **LLM-selected plan** printed above each answer. Both weather and job results remain clearly labeled examples.

In [ ]:
combined_answer = ask("Are any companies hiring in Greensboro? Also, will it be warm there?")

## 10. Enter your own question

Try a phrasing that does **not** use the names of the agents. You can also combine services in one sentence. The LLM—not keyword matching—selects the calls.

In [ ]:
question = "I buy 4 mugs for 85 dollars each and sell them for 120 each. How much do I earn?"
answer = ask(question)

## 11. Optional: inspect the graph

LangGraph can draw the nodes and their possible connections in a notebook.

In [ ]:
from IPython.display import Image, display

try:
    display(Image(assistant.get_graph().draw_mermaid_png()))
except Exception as exc:
    print("Diagram unavailable:", exc)
    print(assistant.get_graph().draw_mermaid())

## 12. Moving toward a production system

1. Connect the weather and jobs tools to real API endpoints (the example requests are commented out).
2. Use a stronger LLM or a model that supports native structured output and tool calling; small local models may occasionally return invalid plans.
3. Validate all model-supplied arguments and ask follow-up questions for missing values instead of inventing them.
4. Add authentication, API error handling, permissions, rate limits, and a maximum number of tool calls.
5. If you need real autonomy, add planner–executor–reviewer steps, feedback loops, and a human approval node.

**Checkpoint:** What is the difference between the **LLM deciding which agents to use** and **LangGraph executing those decisions**?

References: [LangChain](https://docs.langchain.com/oss/python/langchain/overview) · [LangGraph](https://docs.langchain.com/oss/python/langgraph/overview) · [Hugging Face integration](https://reference.langchain.com/python/langchain-huggingface/llms/huggingface_pipeline/HuggingFacePipeline)